In [ ]:
# ================================================================
# ATTENTION-BASED ENGLISH -> TAMIL TRANSLATOR
# Using En-Ta-English.txt and En-Ta-Tamil.txt
# ================================================================


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import os
import re
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from sklearn.model_selection import train_test_split


# ================================================================
# 2. CONFIGURATION
# ================================================================

ENGLISH_FILE = "En-Ta-English.txt"
TAMIL_FILE = "En-Ta-Tamil.txt"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# ================================================================
# 3. SPECIAL TOKENS
# ================================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
START_TOKEN = "<START>"
END_TOKEN = "<END>"

PAD_IDX = 0
UNK_IDX = 1
START_IDX = 2
END_IDX = 3


# ================================================================
# 4. FILE CHECK
# ================================================================

print("\nCurrent folder:")
print(os.getcwd())

print("\nChecking files...")

if not os.path.exists(ENGLISH_FILE):
    raise FileNotFoundError(
        f"{ENGLISH_FILE} not found. "
        "Make sure it is in the same folder as the notebook."
    )

if not os.path.exists(TAMIL_FILE):
    raise FileNotFoundError(
        f"{TAMIL_FILE} not found. "
        "Make sure it is in the same folder as the notebook."
    )

print("English file found:", ENGLISH_FILE)
print("Tamil file found:", TAMIL_FILE)


# ================================================================
# 5. LOAD CORPUS
# ================================================================

with open(
    ENGLISH_FILE,
    "r",
    encoding="utf-8"
) as f:

    english_lines = f.readlines()


with open(
    TAMIL_FILE,
    "r",
    encoding="utf-8"
) as f:

    tamil_lines = f.readlines()


print("\nOriginal English lines:", len(english_lines))
print("Original Tamil lines:", len(tamil_lines))


# ================================================================
# 6. REMOVE CORPUS METADATA
# ================================================================

# The corpus contains metadata at the beginning.
# Remove the first 3 lines.

if len(english_lines) >= 3:
    english_lines = english_lines[3:]

if len(tamil_lines) >= 3:
    tamil_lines = tamil_lines[3:]


print("\nAfter removing metadata:")
print("English lines:", len(english_lines))
print("Tamil lines:", len(tamil_lines))


# ================================================================
# 7. CHECK ALIGNMENT
# ================================================================

if len(english_lines) != len(tamil_lines):

    raise ValueError(
        "English and Tamil files do not contain "
        "the same number of lines."
    )

print("\nEnglish and Tamil files are line-aligned.")
print("Parallel sentence pairs:", len(english_lines))


# ================================================================
# 8. CLEANING FUNCTIONS
# ================================================================

def clean_english(text):

    text = text.lower()

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def clean_tamil(text):

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ================================================================
# 9. TOKENIZATION
# ================================================================

def tokenize_english(sentence):

    return re.findall(
        r"[a-zA-Z]+(?:'[a-zA-Z]+)?|[0-9]+|[^\w\s]",
        sentence
    )


def tokenize_tamil(sentence):

    return sentence.split()


# ================================================================
# 10. CREATE PARALLEL PAIRS
# ================================================================

pairs = []

for english, tamil in zip(
    english_lines,
    tamil_lines
):

    english = clean_english(
        english
    )

    tamil = clean_tamil(
        tamil
    )

    if english and tamil:

        pairs.append(
            (english, tamil)
        )


print("\nValid parallel pairs:", len(pairs))


# ================================================================
# 11. LIMIT SENTENCE LENGTH
# ================================================================

# Smaller sentence length reduces memory usage.

MAX_SOURCE_LENGTH = 20
MAX_TARGET_LENGTH = 20

filtered_pairs = []

for english, tamil in pairs:

    english_tokens = tokenize_english(
        english
    )

    tamil_tokens = tokenize_tamil(
        tamil
    )

    if (
        len(english_tokens) <= MAX_SOURCE_LENGTH
        and
        len(tamil_tokens) <= MAX_TARGET_LENGTH
    ):

        filtered_pairs.append(
            (english, tamil)
        )


pairs = filtered_pairs

print(
    "Pairs after length filtering:",
    len(pairs)
)


# ================================================================
# 12. DISPLAY SAMPLE DATA
# ================================================================

print("\nSample corpus pairs:\n")

for i in range(min(5, len(pairs))):

    english, tamil = pairs[i]

    print("=" * 70)

    print("English:")
    print(english)

    print("\nTamil:")
    print(tamil)


# ================================================================
# 13. TRAIN / VALIDATION / TEST SPLIT
# ================================================================

train_pairs, test_pairs = train_test_split(
    pairs,
    test_size=0.10,
    random_state=SEED
)

train_pairs, valid_pairs = train_test_split(
    train_pairs,
    test_size=0.10,
    random_state=SEED
)


print("\nDataset split:")
print("Training:", len(train_pairs))
print("Validation:", len(valid_pairs))
print("Testing:", len(test_pairs))


# ================================================================
# 14. BUILD VOCABULARY
# ================================================================

def build_vocab(
    pairs,
    language,
    min_frequency=1
):

    counter = Counter()

    for english, tamil in pairs:

        if language == "english":

            tokens = tokenize_english(
                english
            )

        else:

            tokens = tokenize_tamil(
                tamil
            )

        counter.update(tokens)


    vocab = {
        PAD_TOKEN: PAD_IDX,
        UNK_TOKEN: UNK_IDX,
        START_TOKEN: START_IDX,
        END_TOKEN: END_IDX
    }


    for word, frequency in counter.items():

        if frequency >= min_frequency:

            if word not in vocab:

                vocab[word] = len(vocab)


    return vocab


# ================================================================
# 15. CREATE ENGLISH AND TAMIL VOCABULARIES
# ================================================================

english_vocab = build_vocab(
    train_pairs,
    "english"
)

tamil_vocab = build_vocab(
    train_pairs,
    "tamil"
)


print("\nVocabulary sizes:")
print("English vocabulary:", len(english_vocab))
print("Tamil vocabulary:", len(tamil_vocab))


# ================================================================
# 16. REVERSE VOCABULARIES
# ================================================================

english_itos = {
    index: word
    for word, index in english_vocab.items()
}

tamil_itos = {
    index: word
    for word, index in tamil_vocab.items()
}


# ================================================================
# 17. SENTENCE -> NUMERICAL SEQUENCE
# ================================================================

def sentence_to_indices(
    sentence,
    vocabulary,
    language,
    add_start_end=False
):

    if language == "english":

        tokens = tokenize_english(
            sentence
        )

    else:

        tokens = tokenize_tamil(
            sentence
        )


    indices = []


    if add_start_end:

        indices.append(
            vocabulary[START_TOKEN]
        )


    for token in tokens:

        indices.append(
            vocabulary.get(
                token,
                vocabulary[UNK_TOKEN]
            )
        )


    if add_start_end:

        indices.append(
            vocabulary[END_TOKEN]
        )


    return indices


# ================================================================
# 18. DATASET CLASS
# ================================================================

class TranslationDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        pairs,
        english_vocab,
        tamil_vocab
    ):

        self.pairs = pairs

        self.english_vocab = english_vocab

        self.tamil_vocab = tamil_vocab


    def __len__(self):

        return len(self.pairs)


    def __getitem__(self, index):

        english_sentence, tamil_sentence = (
            self.pairs[index]
        )


        english_indices = sentence_to_indices(
            english_sentence,
            self.english_vocab,
            "english",
            add_start_end=False
        )


        tamil_indices = sentence_to_indices(
            tamil_sentence,
            self.tamil_vocab,
            "tamil",
            add_start_end=True
        )


        return (
            torch.tensor(
                english_indices,
                dtype=torch.long
            ),

            torch.tensor(
                tamil_indices,
                dtype=torch.long
            )
        )


# ================================================================
# 19. PADDING / COLLATE FUNCTION
# ================================================================

def collate_fn(batch):

    english_batch = [
        item[0]
        for item in batch
    ]

    tamil_batch = [
        item[1]
        for item in batch
    ]


    english_batch = nn.utils.rnn.pad_sequence(
        english_batch,
        padding_value=PAD_IDX
    )


    tamil_batch = nn.utils.rnn.pad_sequence(
        tamil_batch,
        padding_value=PAD_IDX
    )


    return (
        english_batch.to(DEVICE),
        tamil_batch.to(DEVICE)
    )


# ================================================================
# 20. CREATE DATASETS
# ================================================================

train_dataset = TranslationDataset(
    train_pairs,
    english_vocab,
    tamil_vocab
)

valid_dataset = TranslationDataset(
    valid_pairs,
    english_vocab,
    tamil_vocab
)

test_dataset = TranslationDataset(
    test_pairs,
    english_vocab,
    tamil_vocab
)


print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))
print("Test:", len(test_dataset))


# ================================================================
# 21. DATALOADERS
# ================================================================

BATCH_SIZE = 8


train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)


valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


# ================================================================
# 22. ENCODER
# ================================================================

class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim,
        dropout=0.2
    ):

        super().__init__()


        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim,
            padding_idx=PAD_IDX
        )


        self.rnn = nn.GRU(
            embedding_dim,
            hidden_dim,
            bidirectional=True
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, source):

        embedded = self.dropout(
            self.embedding(source)
        )


        outputs, hidden = self.rnn(
            embedded
        )


        return outputs, hidden


# ================================================================
# 23. ATTENTION
# ================================================================

class Attention(nn.Module):

    def __init__(
        self,
        encoder_hidden_dim,
        decoder_hidden_dim
    ):

        super().__init__()


        self.attention = nn.Linear(
            encoder_hidden_dim * 2
            + decoder_hidden_dim,
            decoder_hidden_dim
        )


        self.v = nn.Linear(
            decoder_hidden_dim,
            1,
            bias=False
        )


    def forward(
        self,
        decoder_hidden,
        encoder_outputs,
        mask
    ):

        source_length = (
            encoder_outputs.shape[0]
        )


        decoder_hidden = (
            decoder_hidden
            .unsqueeze(1)
            .repeat(
                1,
                source_length,
                1
            )
        )


        encoder_outputs = (
            encoder_outputs
            .permute(1, 0, 2)
        )


        energy = torch.tanh(
            self.attention(
                torch.cat(
                    (
                        decoder_hidden,
                        encoder_outputs
                    ),
                    dim=2
                )
            )
        )


        attention = self.v(
            energy
        ).squeeze(2)


        attention = attention.masked_fill(
            mask == 0,
            -1e10
        )


        return torch.softmax(
            attention,
            dim=1
        )


# ================================================================
# 24. DECODER
# ================================================================

class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention,
        dropout=0.2
    ):

        super().__init__()


        self.output_dim = output_dim

        self.attention = attention


        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim,
            padding_idx=PAD_IDX
        )


        self.rnn = nn.GRU(
            embedding_dim
            + encoder_hidden_dim * 2,
            decoder_hidden_dim
        )


        self.fc_out = nn.Linear(
            embedding_dim
            + encoder_hidden_dim * 2
            + decoder_hidden_dim,
            output_dim
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        input_token,
        hidden,
        encoder_outputs,
        mask
    ):

        input_token = (
            input_token.unsqueeze(0)
        )


        embedded = self.dropout(
            self.embedding(input_token)
        )


        attention_weights = self.attention(
            hidden[-1],
            encoder_outputs,
            mask
        )


        attention_weights = (
            attention_weights
            .unsqueeze(1)
        )


        encoder_outputs = (
            encoder_outputs
            .permute(1, 0, 2)
        )


        weighted_encoder = torch.bmm(
            attention_weights,
            encoder_outputs
        )


        weighted_encoder = (
            weighted_encoder
            .permute(1, 0, 2)
        )


        rnn_input = torch.cat(
            (
                embedded,
                weighted_encoder
            ),
            dim=2
        )


        output, hidden = self.rnn(
            rnn_input,
            hidden
        )


        output = output.squeeze(0)

        embedded = embedded.squeeze(0)

        weighted_encoder = (
            weighted_encoder.squeeze(0)
        )


        prediction = self.fc_out(
            torch.cat(
                (
                    output,
                    weighted_encoder,
                    embedded
                ),
                dim=1
            )
        )


        return (
            prediction,
            hidden,
            attention_weights.squeeze(1)
        )


# ================================================================
# 25. SEQ2SEQ MODEL
# ================================================================

class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder,
        device
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device


    def create_mask(self, source):

        return (
            source != PAD_IDX
        ).permute(1, 0)


    def forward(
        self,
        source,
        target,
        teacher_forcing_ratio=0.5
    ):

        batch_size = source.shape[1]

        target_length = target.shape[0]

        target_vocab_size = (
            self.decoder.output_dim
        )


        outputs = torch.zeros(
            target_length,
            batch_size,
            target_vocab_size,
            device=self.device
        )


        encoder_outputs, hidden = (
            self.encoder(source)
        )


        # Combine forward and backward encoder states

        hidden = torch.cat(
            (
                hidden[-2],
                hidden[-1]
            ),
            dim=1
        )


        hidden = hidden.unsqueeze(0)


        mask = self.create_mask(
            source
        )


        input_token = target[0, :]


        # IMPORTANT:
        # No attention history is stored during training.
        # This reduces memory usage significantly.

        for t in range(
            1,
            target_length
        ):

            output, hidden, _ = (
                self.decoder(
                    input_token,
                    hidden,
                    encoder_outputs,
                    mask
                )
            )


            outputs[t] = output


            teacher_force = (
                random.random()
                < teacher_forcing_ratio
            )


            predicted_token = (
                output.argmax(1)
            )


            if teacher_force:

                input_token = target[t]

            else:

                input_token = predicted_token


        return outputs


# ================================================================
# 26. MODEL PARAMETERS
# ================================================================

INPUT_DIM = len(english_vocab)

OUTPUT_DIM = len(tamil_vocab)

# SMALLER MODEL TO PREVENT KERNEL CRASH

ENC_EMB_DIM = 64
DEC_EMB_DIM = 64

ENC_HID_DIM = 128
DEC_HID_DIM = 256


print("\nModel configuration:")
print("Input vocabulary:", INPUT_DIM)
print("Output vocabulary:", OUTPUT_DIM)
print("Encoder embedding:", ENC_EMB_DIM)
print("Decoder embedding:", DEC_EMB_DIM)
print("Encoder hidden:", ENC_HID_DIM)
print("Decoder hidden:", DEC_HID_DIM)


# ================================================================
# 27. CREATE MODEL
# ================================================================

attention = Attention(
    ENC_HID_DIM,
    DEC_HID_DIM
)


encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    ENC_HID_DIM
)


decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    ENC_HID_DIM,
    DEC_HID_DIM,
    attention
)


model = Seq2Seq(
    encoder,
    decoder,
    DEVICE
).to(DEVICE)


# ================================================================
# 28. INITIALIZE WEIGHTS
# ================================================================

def initialize_weights(model):

    for parameter in model.parameters():

        if parameter.dim() > 1:

            nn.init.xavier_uniform_(
                parameter
            )

        else:

            nn.init.zeros_(
                parameter
            )


model.apply(
    initialize_weights
)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)


print(
    "\nTotal trainable parameters:",
    f"{total_parameters:,}"
)


# ================================================================
# 29. OPTIMIZER AND LOSS
# ================================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)


# ================================================================
# 30. TRAIN FUNCTION
# ================================================================

def train(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    epoch_loss = 0


    for source, target in loader:

        optimizer.zero_grad()


        output = model(
            source,
            target,
            teacher_forcing_ratio=0.5
        )


        output_dim = output.shape[-1]


        output = output[1:].reshape(
            -1,
            output_dim
        )


        target = target[1:].reshape(
            -1
        )


        loss = criterion(
            output,
            target
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1
        )


        optimizer.step()


        epoch_loss += loss.item()


    return (
        epoch_loss / len(loader)
    )


# ================================================================
# 31. VALIDATION FUNCTION
# ================================================================

def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    epoch_loss = 0


    with torch.no_grad():

        for source, target in loader:

            output = model(
                source,
                target,
                teacher_forcing_ratio=0
            )


            output_dim = output.shape[-1]


            output = output[1:].reshape(
                -1,
                output_dim
            )


            target = target[1:].reshape(
                -1
            )


            loss = criterion(
                output,
                target
            )


            epoch_loss += loss.item()


    return (
        epoch_loss / len(loader)
    )


# ================================================================
# 32. TRAIN MODEL
# ================================================================

N_EPOCHS = 5

best_validation_loss = float("inf")

train_losses = []

validation_losses = []


print("\n")
print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)


for epoch in range(N_EPOCHS):

    start_time = time.time()


    train_loss = train(
        model,
        train_loader,
        optimizer,
        criterion
    )


    validation_loss = evaluate(
        model,
        valid_loader,
        criterion
    )


    train_losses.append(
        train_loss
    )

    validation_losses.append(
        validation_loss
    )


    elapsed = (
        time.time() - start_time
    )


    print(
        f"Epoch {epoch + 1}/{N_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {validation_loss:.4f} | "
        f"Time: {elapsed:.1f}s"
    )


    if validation_loss < best_validation_loss:

        best_validation_loss = (
            validation_loss
        )


        torch.save(
            model.state_dict(),
            "english_tamil_attention_model.pt"
        )


        print("Best model saved.")


print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)


# ================================================================
# 33. PLOT TRAINING LOSS
# ================================================================

plt.figure(figsize=(10, 5))

plt.plot(
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    validation_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title(
    "English-Tamil Attention Model Training"
)

plt.legend()

plt.grid()

plt.show()


# ================================================================
# 34. LOAD BEST MODEL
# ================================================================

model.load_state_dict(
    torch.load(
        "english_tamil_attention_model.pt",
        map_location=DEVICE
    )
)

model.eval()

print(
    "\nBest model loaded successfully."
)


# ================================================================
# 35. TRANSLATION FUNCTION
# ================================================================

def translate_sentence(
    sentence,
    model,
    max_length=20
):

    model.eval()


    # ------------------------------------------------------------
    # CLEAN INPUT
    # ------------------------------------------------------------

    sentence = clean_english(
        sentence
    )


    # ------------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------------

    tokens = tokenize_english(
        sentence
    )


    # Limit input length

    tokens = tokens[
        :MAX_SOURCE_LENGTH
    ]


    # ------------------------------------------------------------
    # NUMERICAL CONVERSION
    # ------------------------------------------------------------

    source_indices = []

    for token in tokens:

        source_indices.append(
            english_vocab.get(
                token,
                UNK_IDX
            )
        )


    if len(source_indices) == 0:

        return [], [], tokens


    source_tensor = torch.tensor(
        source_indices,
        dtype=torch.long
    ).unsqueeze(1).to(DEVICE)


    # ------------------------------------------------------------
    # ENCODER
    # ------------------------------------------------------------

    with torch.no_grad():

        encoder_outputs, hidden = (
            model.encoder(
                source_tensor
            )
        )


    # Combine forward and backward states

    hidden = torch.cat(
        (
            hidden[-2],
            hidden[-1]
        ),
        dim=1
    )


    hidden = hidden.unsqueeze(0)


    mask = model.create_mask(
        source_tensor
    )


    # ------------------------------------------------------------
    # START DECODING
    # ------------------------------------------------------------

    input_token = torch.tensor(
        [START_IDX],
        dtype=torch.long
    ).to(DEVICE)


    translated_words = []

    attention_scores = []


    # ------------------------------------------------------------
    # GENERATE TAMIL
    # ------------------------------------------------------------

    for _ in range(max_length):

        with torch.no_grad():

            output, hidden, attention = (
                model.decoder(
                    input_token,
                    hidden,
                    encoder_outputs,
                    mask
                )
            )


        predicted_token = (
            output.argmax(1).item()
        )


        # Stop when END token is generated

        if predicted_token == END_IDX:

            break


        # Ignore special tokens

        if predicted_token not in [
            PAD_IDX,
            START_IDX
        ]:

            word = tamil_itos.get(
                predicted_token,
                UNK_TOKEN
            )

            translated_words.append(
                word
            )


        attention_scores.append(
            attention.squeeze(0)
            .cpu()
            .numpy()
        )


        input_token = torch.tensor(
            [predicted_token],
            dtype=torch.long
        ).to(DEVICE)


    return (
        translated_words,
        attention_scores,
        tokens
    )


# ================================================================
# 36. ATTENTION VISUALIZATION
# ================================================================

def plot_attention(
    source_tokens,
    translated_words,
    attention_scores
):

    if len(attention_scores) == 0:

        print(
            "No attention scores available."
        )

        return


    attention_matrix = np.array(
        attention_scores
    )


    # Make dimensions match

    rows = min(
        len(translated_words),
        attention_matrix.shape[0]
    )

    columns = min(
        len(source_tokens),
        attention_matrix.shape[1]
    )


    attention_matrix = (
        attention_matrix[
            :rows,
            :columns
        ]
    )


    words = translated_words[:rows]

    source = source_tokens[:columns]


    plt.figure(
        figsize=(
            max(8, columns * 0.8),
            max(5, rows * 0.5)
        )
    )


    plt.imshow(
        attention_matrix,
        aspect="auto"
    )


    plt.xticks(
        range(len(source)),
        source,
        rotation=45,
        ha="right"
    )


    plt.yticks(
        range(len(words)),
        words
    )


    plt.xlabel(
        "English Source Words"
    )


    plt.ylabel(
        "Generated Tamil Words"
    )


    plt.title(
        "Attention Visualization"
    )


    plt.colorbar(
        label="Attention Weight"
    )


    plt.tight_layout()

    plt.show()


# ================================================================
# 37. TEST ON AN UNSEEN TEST SENTENCE
# ================================================================

print("\n")
print("=" * 70)
print("UNSEEN TEST SENTENCE")
print("=" * 70)


test_english, test_tamil = test_pairs[0]


prediction, attention_scores, source_tokens = (
    translate_sentence(
        test_english,
        model
    )
)


print("\nEnglish:")
print(test_english)


print("\nActual Tamil:")
print(test_tamil)


print("\nPredicted Tamil:")
print(
    " ".join(prediction)
)


# ================================================================
# 38. DISPLAY ATTENTION FOR TEST SENTENCE
# ================================================================

if len(prediction) > 0:

    plot_attention(
        source_tokens,
        prediction,
        attention_scores
    )


# ================================================================
# 39. TEST CUSTOM INPUT
# ================================================================

input_sentence = "How are you?"


prediction, attention_scores, source_tokens = (
    translate_sentence(
        input_sentence,
        model
    )
)


print("\n")
print("=" * 70)
print("CUSTOM INPUT TRANSLATION")
print("=" * 70)


print("\nInput English:")
print(input_sentence)


print("\nGenerated Tamil:")

if len(prediction) > 0:

    print(
        " ".join(prediction)
    )

else:

    print(
        "No translation generated."
    )


# ================================================================
# 40. DISPLAY ATTENTION FOR CUSTOM INPUT
# ================================================================

if len(prediction) > 0:

    plot_attention(
        source_tokens,
        prediction,
        attention_scores
    )


# ================================================================
# 41. INTERACTIVE TRANSLATOR
# ================================================================

print("\n")
print("=" * 70)
print("INTERACTIVE ENGLISH -> TAMIL TRANSLATOR")
print("=" * 70)

print(
    "Enter an English sentence."
)

print(
    "Type 'exit' to stop."
)


while True:

    user_input = input(
        "\nEnglish: "
    )


    if user_input.lower().strip() == "exit":

        print(
            "Translator stopped."
        )

        break


    if not user_input.strip():

        print(
            "Please enter a sentence."
        )

        continue


    prediction, attention_scores, source_tokens = (
        translate_sentence(
            user_input,
            model
        )
    )


    print(
        "\nTamil:"
    )


    if len(prediction) > 0:

        print(
            " ".join(prediction)
        )

    else:

        print(
            "No translation generated."
        )


    if len(prediction) > 0:

        plot_attention(
            source_tokens,
            prediction,
            attention_scores
        )


# ================================================================
# END
# ================================================================

print("\nProject completed.")
print(
    "Model saved as: "
    "english_tamil_attention_model.pt"
)

Device: cpu

Current folder:
C:\Users\kirut\OneDrive\KEERTHI\semester-5\NLP Skill\Class Task

Checking files...
English file found: En-Ta-English.txt
Tamil file found: En-Ta-Tamil.txt

Original English lines: 8949
Original Tamil lines: 8949

After removing metadata:
English lines: 8946
Tamil lines: 8946

English and Tamil files are line-aligned.
Parallel sentence pairs: 8946

Valid parallel pairs: 8899
Pairs after length filtering: 7707

Sample corpus pairs:

English:
ranaviru sewa authority

Tamil:
ரணவிரு சேவை அதிகார சபை
English:
annual report 2011

Tamil:
வருடாந்த அறிக்கை 2011
English:
no.301, 4th floor,

Tamil:
இல. 301, 4ஆம் மாடி,
English:
t.b.jayah mawatha

Tamil:
டி.பி. ஜாயா மாவத்தை
English:
colombo 10

Tamil:
கொழும்பு 10

Dataset split:
Training: 6242
Validation: 694
Testing: 771

Vocabulary sizes:
English vocabulary: 4098
Tamil vocabulary: 7765

Dataset sizes:
Train: 6242
Validation: 694
Test: 771

Model configuration:
Input vocabulary: 4098
Output vocabulary: 7765
Encoder embed